# Plotting Semantic Primitives

PyTex plotting now treats semantic objects as the plotting inputs rather than
expecting users to manually convert everything into anonymous arrays.

The runtime plotting surface returns normal Matplotlib figures. Canonical
documentation assets may still be exported to SVG when they belong in the repo.


In [ ]:
from pathlib import Path
import tempfile

import numpy as np

from pytex import (
    AcquisitionGeometry,
    AtomicSite,
    BenchmarkManifest,
    build_crystal_scene,
    CalibrationRecord,
    CrystalCellOverlay,
    CrystalDirection,
    CrystalDirectionOverlay,
    CrystalMap,
    CrystalPlane,
    CrystalPlaneOverlay,
    DirectionAnnotationStyle,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    get_phase_fixture,
    list_phase_fixtures,
    list_style_themes,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    PowderPattern,
    PowderReflection,
    ReferenceFrame,
    read_validation_manifest,
    read_workflow_result_manifest,
    resolve_style,
    RadiationSpec,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    UnitCell,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    PlaneAnnotationStyle,
    generate_saed_pattern,
    generate_xrd_pattern,
    plot_odf,
    plot_crystal_structure_3d,
    plot_inverse_pole_figure,
    plot_orientations,
    plot_pole_figure,
    plot_saed_pattern,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
    plot_xrd_pattern,
)


def make_crystal_frame():
    return ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )


def make_context():
    crystal = make_crystal_frame()
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    phase = get_phase_fixture("ni_fcc").load_phase(crystal_frame=crystal)
    return crystal, specimen, map_frame, detector, lab, phase


def describe_phase_fixture(fixture_id):
    record = get_phase_fixture(fixture_id)
    return {
        "fixture_id": record.fixture_id,
        "display_name": record.display_name,
        "artifact_path": str(record.artifact_path),
        "metadata_path": str(record.metadata_path),
        "intended_uses": tuple(record.metadata["intended_uses"]),
    }


def load_zr_hcp_phase():
    return get_phase_fixture("zr_hcp").load_phase(crystal_frame=make_crystal_frame())


def load_diamond_phase():
    return get_phase_fixture("diamond").load_phase(crystal_frame=make_crystal_frame())


def publication_crystal_style():
    return {
        "crystal": {
            "atom_radius_scale": 0.5,
            "atom_edgewidth": 0.0,
            "atom_surface_resolution": 34,
            "bond_surface_resolution": 28,
            "bond_alpha": 0.72,
            "bond_color": "#7c8ea3",
            "atom_specular_strength": 0.42,
            "light_specular": 0.4,
        }
    }


In [ ]:
crystal, specimen, map_frame, detector, lab, phase = make_context()

vectors = VectorSet(
    values=np.array(
        [
            [1.0, 0.0, 0.0],
            [0.0, 1.0, 0.0],
            [0.0, 0.0, 1.0],
        ]
    ),
    reference_frame=crystal,
)
figure = plot_vector_set(vectors, normalize=True)
figure


In [ ]:
symmetry = phase.symmetry
orbit_figure = plot_symmetry_orbit(symmetry, np.array([1.0, 0.0, 0.0]))
elements_figure = plot_symmetry_elements(symmetry)
orbit_figure


In [ ]:
orientations = OrientationSet.from_euler_angles(
    np.array(
        [
            [0.0, 0.0, 0.0],
            [45.0, 35.0, 10.0],
            [90.0, 0.0, 0.0],
        ]
    ),
    crystal_frame=crystal,
    specimen_frame=specimen,
    symmetry=symmetry,
    phase=phase,
)
orientation_figure = plot_orientations(orientations, representation="euler")
orientation_figure


In [ ]:
odf = ODF.from_orientations(orientations, weights=[4.0, 2.0, 1.0])
pole = CrystalPlane(miller=MillerIndex([1, 0, 0], phase=phase), phase=phase)
pole_figure = odf.reconstruct_pole_figure(
    pole,
    include_symmetry_family=False,
    antipodal=False,
)
ipf = InversePoleFigure.from_orientations(
    orientations,
    [0.0, 0.0, 1.0],
    reduce_by_symmetry=True,
    antipodal=True,
)
pf_figure = plot_pole_figure(
    pole_figure,
    kind="contour",
    bins=81,
    sigma_bins=1.5,
    levels=12,
)
ipf_figure = plot_inverse_pole_figure(ipf)
odf_figure = plot_odf(
    odf,
    kind="sections",
    section_phi2_deg=(0.0, 30.0, 60.0),
    section_phi1_steps=91,
    section_big_phi_steps=46,
    levels=10,
)
pf_figure


In [ ]:
ipf_figure


In [ ]:
odf_figure


The plotting notebook now shows the three main texture inspection surfaces
implemented today:

- contour pole figures
- inverse pole figures
- Bunge-section ODF plots

All of them are produced by the same runtime plotting API that ordinary user code
calls, rather than by notebook-specific plotting helpers.
